# Chapter 2: When Simple Tools Aren't Enough

This notebook covers the advanced patterns from Chapter 2:

1. Class-based tools — sharing resources across tools
2. Async tools — parallel execution for slow API calls

In [2]:
%pip install strands-agents -q

/home/agent/.venvs/ai-agents-ch1-5/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


---
## 1. The Database Connection Problem

When each `@tool` function opens its own database connection, you get 50 connections/minute in production.

**The fix:** Group related tools in a class. Connect once in `__init__`, share via `self`.

The example below uses a simple dict to simulate a shared data store.
In production, replace `self.products` with a real database connection.

In [3]:
from strands import Agent, tool

class InventoryTools:
    def __init__(self):
        # Shared resource: all tools access the same data store.
        # In production: self.db = connect_to_database()
        self.products = {
            "PROD-123": {"name": "Wireless Mouse", "quantity": 15, "price": 29.99},
            "PROD-456": {"name": "USB-C Hub", "quantity": 0, "price": 49.99},
            "PROD-789": {"name": "Mechanical Keyboard", "quantity": 8, "price": 89.99},
        }

    @tool
    def check_stock(self, product_id: str) -> str:
        """Check product stock level.

        Args:
            product_id: The product ID to check
        """
        product = self.products.get(product_id)
        if product:
            return f"{product['name']} ({product_id}): {product['quantity']} units in stock"
        return f"Product {product_id} not found"

    @tool
    def update_stock(self, product_id: str, quantity: int) -> str:
        """Update product stock quantity.

        Args:
            product_id: The product ID to update
            quantity: New quantity to set
        """
        if product_id in self.products:
            self.products[product_id]["quantity"] = quantity
            return f"Updated {product_id} to {quantity} units"
        return f"Product {product_id} not found"

# One instance, shared state, multiple tools
inventory = InventoryTools()
agent = Agent(tools=[inventory.check_stock, inventory.update_stock])

agent("Check stock for PROD-123")
agent("Update PROD-456 stock to 25 units, then confirm the new level")

I'll check the stock for PROD-123 right away!
Tool #1: check_stock
Here are the stock details for **PROD-123**:

- **Product:** Wireless Mouse
- **Product ID:** PROD-123
- **Stock Level:** 15 units in stock

Let me know if you'd like to make any updates or need further assistance!I'll update the stock for PROD-456 to 25 units right away!
Tool #2: update_stock
Stock updated successfully! Now let me confirm the new stock level.
Tool #3: check_stock
Here's the confirmation for **PROD-456**:

- **Product:** USB-C Hub
- **Product ID:** PROD-456
- **Updated Stock Level:** ✅ 25 units in stock

The stock has been successfully updated and confirmed! Let me know if there's anything else you need.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's the confirmation for **PROD-456**:\n\n- **Product:** USB-C Hub\n- **Product ID:** PROD-456\n- **Updated Stock Level:** ✅ 25 units in stock\n\nThe stock has been successfully updated and confirmed! Let me know if there's anything else you need."}], 'metadata': {'usage': {'inputTokens': 1067, 'outputTokens': 72, 'totalTokens': 1139}, 'metrics': {'latencyMs': 1653, 'timeToFirstByteMs': 1976}}, 'tracking_id': '672f31fb-74bd-4830-beed-3e2f10af8d88'}, metrics=EventLoopMetrics(cycle_count=5, tool_metrics={'check_stock': ToolMetrics(tool={'toolUseId': 'tooluse_NHyxwLap6MhAW0gDuQ0y4V', 'name': 'check_stock', 'input': {'product_id': 'PROD-456'}}, call_count=2, success_count=2, error_count=0, total_time=0.0008296966552734375), 'update_stock': ToolMetrics(tool={'toolUseId': 'tooluse_G4AUFXh3HHQUPCqSG8r9OL', 'name': 'update_stock', 'input': {'product_id': 'PROD-456', 'quantity': 25}}, call_count=1, succes

---
## 2. Slow Warehouse Checks — Async Tools

Checking 3 warehouses sequentially takes 6 seconds. With async, they run in parallel — ~2 seconds total.

Mark the function as `async`, use `await`, and call the agent with `invoke_async`.

In [4]:
import asyncio
import time
from strands import Agent, tool

@tool
async def check_warehouse_inventory(product_id: str, warehouse: str) -> dict:
    """Check inventory at a specific warehouse.

    Args:
        product_id: Product ID to check
        warehouse: Warehouse identifier (e.g., "east", "west", "central")
    """
    # Simulate API call delay
    await asyncio.sleep(2)

    # Simulated warehouse data
    data = {
        "east":    {"PROD-123": 45, "PROD-456": 12},
        "west":    {"PROD-123": 30, "PROD-456": 0},
        "central": {"PROD-123": 60, "PROD-456": 25},
    }

    quantity = data.get(warehouse, {}).get(product_id, 0)
    return {
        "warehouse": warehouse,
        "product_id": product_id,
        "quantity": quantity
    }

async def main():
    agent = Agent(tools=[check_warehouse_inventory])
    start = time.time()
    response = await agent.invoke_async(
        "Can we ship 100 units of PROD-123? Check all warehouses: east, west, and central."
    )
    elapsed = time.time() - start
    print(response.message['content'][0]['text'])
    print(f"\nTotal time: {elapsed:.1f}s (sequential would be ~6s)")

await main()

Sure! Let me check the inventory for **PROD-123** across all three warehouses simultaneously right away!
Tool #1: check_warehouse_inventory

Tool #2: check_warehouse_inventory

Tool #3: check_warehouse_inventory
Here's a summary of the inventory for **PROD-123** across all warehouses:

| Warehouse | Available Stock |
|-----------|----------------|
| East      | 45 units        |
| West      | 30 units        |
| Central   | 60 units        |
| **Total** | **135 units**   |

✅ **Yes, you can ship 100 units!** The combined inventory across all warehouses totals **135 units**, which is sufficient to fulfill the order of 100 units. No single warehouse can cover the full order on its own, but by combining stock — for example, **60 from Central + 40 from East** — you can meet the requirement with 35 units to spare.Here's a summary of the inventory for **PROD-123** across all warehouses:

| Warehouse | Available Stock |
|-----------|----------------|
| East      | 45 units        |
| West    